# Standardize Manual S&P Mining Export

This notebook converts the manually exported S&P Global mining `.xls` file into a DuckDB database with a small relational schema that can also be populated by a richer future scrape.

The design keeps three layers:

- `properties`: one standardized row per mine/property.
- `property_texts`: verbatim long-form text fields, so no information is lost.
- `property_work_history_events`: parsed dated snippets extracted from `full_work_history_raw` when possible.

That split makes the messy last column usable in two ways at once: the raw text remains available for audit or re-parsing, while a derived event table supports downstream analysis and interchangeability with a fuller relational scrape.

## Dependencies

This notebook expects:

- `pandas`
- `duckdb`
- `python-calamine` or `xlrd` for legacy `.xls` reading
- `openai` for the LLM enrichment step

For the OpenAI enrichment cells, you also need an `OPENAI_API_KEY` environment variable in the notebook session.


In [ ]:
from __future__ import annotations

import json
import os
import re
from pathlib import Path

import duckdb
import pandas as pd
import geopandas as gpd
from openai import OpenAI

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)


In [ ]:
Path.cwd().resolve()

In [ ]:
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "data").exists():
    for candidate in [PROJECT_ROOT.parent, PROJECT_ROOT.parent.parent, PROJECT_ROOT.parent.parent.parent]:
        if (candidate / "data").exists():
            PROJECT_ROOT = candidate
            break

SOURCE_PATH = PROJECT_ROOT / "data/snf_mining/raw/SPGlobal_Export_3-31-2026_d895a56a-e8e3-4698-a5c7-b2931645b812.xls"
OUTPUT_DIR = PROJECT_ROOT / "data/snf_mining/processed/stage_0" / "manual_xls"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DB_PATH = OUTPUT_DIR / "snf_mining_manual_export.duckdb"

SOURCE_PATH, DB_PATH

In [ ]:
conn = duckdb.connect(DB_PATH)
conn.sql("LOAD spatial;")

In [ ]:
# list all tables
conn.sql("SELECT * FROM properties").df()

---

In [ ]:
test = conn.sql("SELECT ST_AsWKT(ST_BUFFER(ST_POINT(longitude, latitude), 1)) FROM properties LIMIT 1").df()

In [ ]:
conn.close()

In [ ]:
gpd.GeoSeries.from_wkt(test.iloc[:, 0]).plot()

In [ ]:
# Simplify ADM2
conn = duckdb.connect()
conn.sql("LOAD spatial;")

In [ ]:
conn.sql("SELECT layers[1] FROM ST_Read_Meta('C:\\Users\\schulz0022\\Documents\\gnt\\data\\misc\\gadm\\raw\\gadm_410-gpkg\\gadm_410.gpkg')").fetchone()

In [ ]:
conn.sql("SELECT * FROM ST_READ('C:\\Users\\schulz0022\\Documents\\gnt\\data\\misc\\gadm\\raw\\gadm_410-gpkg\\gadm_410.gpkg') LIMIT 10")

In [ ]:
, layer=''

In [ ]:
adm2 = conn.sql("SELECT id, name, ST_SimplifyPreserveTopology(geometry, 0.01) AS geometry FROM adm2").df()
conn.close()